In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Optimization Solver : une fonction Qiskit de Q-CTRL Fire Opal
> **Note:** Les fonctions Qiskit sont une fonctionnalité expérimentale réservée aux utilisateurs des plans IBM Quantum&reg; Premium, Flex et On-Prem (via l'API IBM Quantum Platform). Elles sont en version préliminaire et susceptibles d'évoluer.

## Vue d'ensemble
Avec le Fire Opal Optimization Solver, tu peux résoudre des problèmes d'optimisation à l'échelle utilitaire sur du matériel quantique, sans avoir besoin d'expertise en informatique quantique. Il suffit de fournir la définition du problème de haut niveau — le Solver s'occupe du reste. L'ensemble du flux de travail est sensible au bruit et exploite [la gestion des performances Fire Opal](/guides/q-ctrl-performance-management) en arrière-plan. Le Solver fournit de manière constante des solutions précises à des problèmes difficiles à résoudre classiquement, même à l'échelle complète d'un appareil sur les plus grands QPU IBM&reg;.

Le Solver est flexible et peut être utilisé pour résoudre des problèmes d'optimisation combinatoire définis comme des fonctions objectif ou des graphes arbitraires. Les problèmes n'ont pas à être mappés à la topologie de l'appareil. Les problèmes non contraints et contraints sont résolubles, à condition que les contraintes puissent être formulées comme des termes de pénalité. Les exemples de ce guide montrent comment résoudre un problème d'optimisation non contraint et un problème contraint à l'échelle utilitaire en utilisant différents types d'entrées du Solver. Le premier exemple porte sur un problème Max-Cut défini sur un graphe 3-régulier à 156 nœuds, tandis que le second aborde un problème de couverture minimale de sommets (Minimum Vertex Cover) à 50 nœuds défini par une fonction de coût.

Pour accéder à l'Optimization Solver, [contacte Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Description de la fonction
Le Solver optimise et automatise entièrement l'algorithme, depuis la suppression des erreurs au niveau matériel jusqu'au mappage efficace du problème et à l'optimisation classique en boucle fermée. En coulisses, le pipeline du Solver réduit les erreurs à chaque étape, permettant d'atteindre les performances nécessaires pour monter en charge de façon significative. Le flux de travail sous-jacent s'inspire de l'algorithme QAOA (Quantum Approximate Optimization Algorithm), un algorithme hybride quantique-classique. Pour un résumé détaillé du flux de travail complet de l'Optimization Solver, consulte [le manuscrit publié](https://arxiv.org/abs/2406.01743).

![Visualisation du flux de travail de l'Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Pour résoudre un problème générique avec l'Optimization Solver :
1. Définis ton problème sous forme de fonction objectif, de graphe ou de chaîne de spin `SparsePauliOp`.
2. Connecte-toi à la fonction via le catalogue de fonctions Qiskit.
3. Lance le problème avec le Solver et récupère les résultats.
## Entrées et sorties
### Entrées
| Nom    | Type                    | Description                                                                                                                                                                                         | Requis | Défaut | Exemple |
| ---------  |-------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------| -------- | ---------- | ---------- |
| problem  | `str` ou `SparsePauliOp` | L'une des représentations listées sous « Formats de problèmes acceptés »                                                                                                                                  | Oui | N/A   |`Poly(2.0*n[0]*n[1] + 2.0*n[0]*n[2] - 3.13520241298341*n[0] + 2.0*n[1]*n[2] - 3.14469748506794*n[1] - 3.18897660121566*n[2] + 6.0, n[0], n[1], n[2], domain='RR')` |
| problem_type  | `str`                   | Nom de la classe du problème ; utilisé uniquement pour les définitions de graphes et de chaînes de spin, limitées à « maxcut » ou « spin_chain » ; non requis pour les fonctions objectif arbitraires | Non      | `None`| `"maxcut"`      |
| backend_name  | `str`                   | Nom du backend à utiliser                                                                                                                                                                          | Non  | Le backend le moins occupé auquel ton instance a accès    | `"ibm_fez"`      |
| options  | `dict`                  | Options d'entrée, dont : (Optionnel) `session_id` : `str` ; par défaut, crée une nouvelle session                                                                                      | Non | `None`    | `{"session_id": "cw2q70c79ws0008z6g4g"}`     |

**Formats de problèmes acceptés :**
- Représentation polynomiale d'une fonction objectif. Idéalement créée en Python à partir d'un objet SymPy Poly existant et mise en chaîne avec [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Représentation par graphe d'un type de problème spécifique. Le graphe doit être créé avec la bibliothèque networkx en Python, puis converti en chaîne via la fonction networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Représentation en chaîne de spin d'un problème spécifique. La chaîne de spin doit être représentée sous la forme d'un objet `SparsePauliOp` ; consulte la [documentation](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) pour plus de détails.

**Backends pris en charge :**
Exécute le code suivant pour voir la liste des backends actuellement pris en charge. Si ton appareil n'est pas listé, [contacte Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) pour en demander la prise en charge.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

service.backends()

[<IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_brisbane')>,
 <IBMBackend('ibm_pittsburgh')>,
 <IBMBackend('ibm_kingston')>,
 <IBMBackend('ibm_torino')>,
 <IBMBackend('ibm_marrakesh')>]

**Options:**
| Name   | Type   | Description  | Default |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | An existing Qiskit Runtime session ID  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | A list of job tags | `[]`|

### Outputs
| Name    | Type | Description | Example |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | Solution and metadata listed under "Result dictionary contents"         | `{'solution_bitstring_cost': 3.0, ‘final_bitstring_distribution’: {‘000001’: 100, ‘000011’: 2}, ‘iteration_count’: 3, 'solution_bitstring': ‘000001’,  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |


**Result dictionary contents:**
| Name    | Type | Description |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | The best minimum cost across all iterations        |
| final_bitstring_distribution  | `CountsDict`              | The bitstring counts dictionary associated with the minimum cost        |
| solution_bitstring | `str`              | Bitstring with the best cost in the final distribution         |
| iteration_count  | `int`              | The total number of QAOA iterations performed by the Solver        |
| variables_to_bitstring_index_map  | `float`              | The mapping from the variables to the equivalent bit in the bitstring       |
| best_parameters  | `list[float]`            | The optimized beta and gamma parameters across all iterations  |
| warnings  |`list[str]`              | The warnings produced while compiling or running QAOA (defaults to None)   |

## Benchmarks

[Published benchmarking results](https://arxiv.org/abs/2406.01743) show that the Solver successfully solves problems with over 120 qubits, even outperforming previously published results on quantum annealing and trapped-ion devices. The following benchmark metrics provide a rough indication of the accuracy and scaling of problem types based on a few examples. Actual metrics may differ based on various problem features, such as the number of terms in the objective function (density) and their locality, number of variables, and polynomial order.

The "Number of qubits" indicated is not a hard limitation but represents rough thresholds where you can expect extremely consistent solution accuracy. Larger problem sizes have been successfully solved, and testing beyond these limits is encouraged.

Arbitrary qubit connectivity is supported across all problem types.

| Problem type    | Number of qubits | Example | Accuracy | Total time (s) | Runtime usage (s) | Number of iterations
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Sparsely-connected quadratic problems  | 156 | 3-regular Max-Cut| 100%     | 1764     | 293          | 16 |
| Higher-order binary optimization | 156 | Ising spin-glass model | 100%      | 1461     | 272          | 16 |
| Densely-connected quadratic problems | 50 | Fully-connected Max-Cut| 100%      |  1758    | 268  | 12 |
| Constrained problem with penalty terms | 50 | Weighted Minimum Vertex Cover with 8% edge density | 100%      | 1074     | 215 | 10 |

## Get started

First, authenticate using your [IBM Quantum API key](http://quantum.cloud.ibm.com/). Then, select the Qiskit Function as follows. (This snippet assumes you've already [saved your account](/docs/guides/functions#install-qiskit-functions-catalog-client) to your local environment.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

**Options :**
| Nom   | Type   | Description  | Défaut |
|--------|----------|-----------------------|---------------------|
| session_id | `str`  | Identifiant d'une session Qiskit Runtime existante  | `"cw4r3je6f0t010870y3g"` |
| job_tags | `list[str]` | Liste d'étiquettes de tâches | `[]`|

### Sorties
| Nom    | Type | Description | Exemple |
| ---------  | ---------------- | -------------------------- | -------- |
| result  | `dict[str, Any]`              | Solution et métadonnées listées sous « Contenu du dictionnaire de résultats »         | `{'solution_bitstring_cost': 3.0, 'final_bitstring_distribution': {'000001': 100, '000011': 2}, 'iteration_count': 3, 'solution_bitstring': '000001',  'variables_to_bitstring_index_map': {n[1]': 5, 'n[2]': 4, 'n[3]': 3, 'n[4]': 2, 'n[5]': 1}, 'best_parameters': [0.19628831763697097, -1.047052334523102], 'warnings': []}`     |

**Contenu du dictionnaire de résultats :**
| Nom    | Type | Description |
| ---------  | ---------------- | -------------------------- |
| solution_bitstring_cost  | `float`              | Le meilleur coût minimum sur toutes les itérations        |
| final_bitstring_distribution  | `CountsDict`              | Le dictionnaire de comptage de chaînes de bits associé au coût minimum        |
| solution_bitstring | `str`              | La chaîne de bits avec le meilleur coût dans la distribution finale         |
| iteration_count  | `int`              | Le nombre total d'itérations QAOA effectuées par le Solver        |
| variables_to_bitstring_index_map  | `float`              | La correspondance entre les variables et le bit équivalent dans la chaîne de bits       |
| best_parameters  | `list[float]`            | Les paramètres bêta et gamma optimisés sur toutes les itérations  |
| warnings  |`list[str]`              | Les avertissements produits lors de la compilation ou de l'exécution de QAOA (par défaut None)   |
## Benchmarks
Les [résultats de benchmarking publiés](https://arxiv.org/abs/2406.01743) montrent que le Solver résout avec succès des problèmes dépassant 120 qubits, surpassant même des résultats précédemment publiés sur des recuits quantiques et des appareils à ions piégés. Les métriques de benchmark suivantes donnent une indication approximative de la précision et de la mise à l'échelle des types de problèmes, sur la base de quelques exemples. Les métriques réelles peuvent varier selon les caractéristiques du problème, comme le nombre de termes dans la fonction objectif (densité) et leur localité, le nombre de variables et l'ordre polynomial.

Le « Nombre de qubits » indiqué n'est pas une limite absolue, mais représente des seuils approximatifs en dessous desquels tu peux t'attendre à une précision de solution extrêmement constante. Des tailles de problèmes plus importantes ont été résolues avec succès, et les tests au-delà de ces limites sont encouragés.

La connectivité arbitraire entre qubits est prise en charge pour tous les types de problèmes.

| Type de problème    | Nombre de qubits | Exemple | Précision | Temps total (s) | Utilisation du runtime (s) | Nombre d'itérations
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Problèmes quadratiques faiblement connectés  | 156 | Max-Cut 3-régulier | 100 %     | 1764     | 293          | 16 |
| Optimisation binaire d'ordre supérieur | 156 | Modèle de verre de spin d'Ising | 100 %      | 1461     | 272          | 16 |
| Problèmes quadratiques densément connectés | 50 | Max-Cut entièrement connecté | 100 %      |  1758    | 268  | 12 |
| Problème contraint avec termes de pénalité | 50 | Couverture minimale de sommets pondérée avec 8 % de densité d'arêtes | 100 %      | 1074     | 215 | 10 |
## Premiers pas
Commence par t'authentifier avec ta [clé API IBM Quantum](http://quantum.cloud.ibm.com/). Sélectionne ensuite la fonction Qiskit comme suit. (Cet extrait suppose que tu as déjà [enregistré ton compte](/guides/functions#install-qiskit-functions-catalog-client) dans ton environnement local.)

In [2]:
# %pip install networkx numpy

## Exemple : optimisation non contrainte
Lance le problème de [coupe maximale](https://en.wikipedia.org/wiki/Maximum_cut) (Max-Cut). L'exemple suivant illustre les capacités du Solver sur un problème Max-Cut avec un graphe non pondéré 3-régulier à 156 nœuds, mais tu peux aussi résoudre des problèmes de graphes pondérés.
En plus de `qiskit-ibm-catalog`, tu utiliseras également les packages suivants pour cet exemple : `networkx` et `numpy`. Tu peux les installer en décommentant la cellule suivante si tu exécutes cet exemple dans un notebook avec le noyau IPython.

In [2]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [3]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

The Solver accepts a string as the problem definition input.

In [4]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

![Sortie de la cellule de code précédente](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif)

Le Solver accepte une chaîne de caractères comme entrée pour la définition du problème.

In [9]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [ ]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [14]:
# Get job status
print(maxcut_job.status())

QUEUED


Vérifie le [statut](/guides/functions#check-job-status) de ta charge de travail de fonction Qiskit ou récupère les [résultats](/guides/functions#retrieve-results) comme suit :

In [15]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 209.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior Max-Cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [ ]:
# %pip install numpy networkx sympy

### 3. Récupérer le résultat
Récupère la valeur de coupe optimale depuis le dictionnaire de résultats.

> **Note:** La correspondance des variables vers la chaîne de bits peut avoir changé. Le dictionnaire de sortie contient un sous-dictionnaire `variables_to_bitstring_index_map` qui permet de vérifier l'ordre.

In [ ]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

A standard optimization model for weighted MVC can be formulated as follows. First, a penalty must be added for any case where an edge is not connected to a vertex in the subset. Therefore, let $n_i = 1$ if vertex $i$ is in the cover (i.e., in the subset) and $n_i = 0$ otherwise. Second, the goal is to minimize the total number of vertices in the subset, which can be represented by the following function:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

Tu peux vérifier la précision du résultat en résolvant le problème classiquement avec des solveurs open-source comme [PuLP](https://coin-or.github.io/pulp/) si le graphe n'est pas densément connecté. Les problèmes à haute densité peuvent nécessiter des solveurs classiques avancés pour valider la solution.
## Exemple : optimisation contrainte
L'exemple Max-Cut précédent est un problème d'optimisation binaire quadratique non contraint classique. L'Optimization Solver de Q-CTRL peut être utilisé pour divers types de problèmes, y compris l'optimisation contrainte. Tu peux résoudre des types de problèmes arbitraires en fournissant la définition du problème sous forme de polynôme où les contraintes sont modélisées comme des termes de pénalité.

L'exemple suivant montre comment construire une fonction de coût pour un problème d'optimisation contraint, la [couverture minimale de sommets](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
En plus des packages `qiskit-ibm-catalog` et `qiskit`, tu utiliseras également les packages suivants : `numpy`, `networkx` et `sympy`. Tu peux les installer en décommentant la cellule suivante si tu exécutes cet exemple dans un notebook avec le noyau IPython.

In [ ]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

### 1. Définir le problème
Définis un problème MVC aléatoire en générant un graphe avec des nœuds pondérés aléatoirement.

In [ ]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

![Sortie de la cellule de code précédente](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif)

Un modèle d'optimisation standard pour le MVC pondéré peut être formulé comme suit. D'abord, une pénalité doit être ajoutée pour tout cas où une arête n'est pas connectée à un sommet du sous-ensemble. Soit donc $n_i = 1$ si le sommet $i$ est dans la couverture (c'est-à-dire dans le sous-ensemble) et $n_i = 0$ sinon. L'objectif est de minimiser le nombre total de sommets dans le sous-ensemble, ce qui peut être représenté par la fonction suivante :

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [ ]:
print(mvc_job.status())

Désormais, chaque arête du graphe doit inclure au moins un point d'extrémité de la couverture, ce qui peut s'exprimer par l'inégalité :

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Tout cas où une arête n'est pas connectée au sommet de la couverture doit être pénalisé. Cela peut être représenté dans la fonction de coût en ajoutant une pénalité de la forme $P(1-n_i-n_j+n_i n_j)$ où $P$ est une constante de pénalité positive. Ainsi, une alternative non contrainte à l'inégalité contrainte pour le MVC pondéré est :

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [ ]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


### 2. Lancer le problème